In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
scheme_df = pd.read_csv("../data/processed/07_scheme_performance_cleaned.csv")
nav_df = pd.read_csv("../data/processed/02_nav_history_cleaned.csv")

In [3]:
nav_df["date"] = pd.to_datetime(nav_df["date"])

nav_pivot = nav_df.pivot(
    index="date",
    columns="amfi_code",
    values="nav"
)

returns_df = nav_pivot.pct_change().dropna()

returns_df.head()

amfi_code,100016,100025,100033,101206,101207,101208,102885,102886,102887,118632,...,120843,120844,125497,125498,148567,148568,148569,149322,149323,149324
date,,,,,,,,,,,,,,,,,,,,,
2022-01-04,-0.010306,-0.003553,-0.013328,0.001153,-0.010865,-0.000141,0.011122,0.011314,-0.010459,-0.000714,...,0.018160,0.000243,0.001001,-0.005010,0.019318,0.015865,0.008378,-0.008592,0.000482,-0.010498
2022-01-05,0.012865,-0.000050,-0.004386,0.003866,0.000603,0.000382,-0.007878,0.006779,-0.005308,0.005913,...,-0.012304,0.000462,0.004250,-0.005184,-0.003939,-0.007660,0.015294,-0.006480,0.008670,-0.002593
2022-01-06,-0.011377,-0.001880,-0.005167,-0.002128,-0.029101,-0.000143,0.015277,0.031127,0.012678,0.003540,...,0.008509,0.000650,-0.003589,-0.002706,0.011892,-0.004709,0.000863,-0.003818,-0.013861,-0.008382
2022-01-07,-0.001210,0.002036,-0.005748,-0.006314,0.024766,0.000215,-0.009369,-0.008835,-0.016498,-0.005793,...,-0.013477,0.000176,-0.002719,-0.012660,0.000515,0.007001,0.001173,-0.004069,0.004317,0.011680
2022-01-10,-0.008639,0.006791,0.006277,0.011548,0.001251,0.000690,-0.001202,-0.000722,-0.011593,0.006360,...,-0.002583,0.000853,0.003057,-0.019536,0.024234,-0.011127,0.009372,0.001601,0.003650,0.001356


# Fund Performance Analytics

This notebook contains the implementation of:
- CAGR
- Sharpe Ratio
- Sortino Ratio
- Alpha
- Beta
- Tracking Error
- Maximum Drawdown

In [4]:
cagr = ((nav_pivot.iloc[-1] / nav_pivot.iloc[0]) ** (252 / len(nav_pivot))) - 1

cagr_df = pd.DataFrame({
    "CAGR": cagr
})

cagr_df.head()

,CAGR
amfi_code,
100016,0.025412
100025,0.042948
100033,0.288994
101206,0.226048
101207,0.076433


In [5]:
cagr_df.to_csv(
    "../reports/cagr_report.csv"
)

In [6]:
risk_free_rate = 0.06

sharpe_ratio = (
    (returns_df.mean() * 252 - risk_free_rate)
    /
    (returns_df.std() * np.sqrt(252))
)

sharpe_df = pd.DataFrame({
    "Sharpe_Ratio": sharpe_ratio
})

sharpe_df.head()

,Sharpe_Ratio
amfi_code,
100016,-0.167148
100025,-0.439062
100033,1.120102
101206,1.061535
101207,0.182043


In [7]:
sharpe_df.to_csv(
    "../reports/sharpe_ratio_report.csv"
)

In [8]:
downside_returns = returns_df.copy()
downside_returns[downside_returns > 0] = 0

sortino_ratio = (
    (returns_df.mean() * 252 - risk_free_rate)
    /
    (downside_returns.std() * np.sqrt(252))
)

sortino_df = pd.DataFrame({
    "Sortino_Ratio": sortino_ratio
})

sortino_df.head()

,Sortino_Ratio
amfi_code,
100016,-0.296696
100025,-0.781696
100033,2.014810
101206,1.952710
101207,0.319604


In [9]:
sortino_df.to_csv(
    "../reports/sortino_ratio_report.csv"
)

In [10]:
cumulative_returns = (1 + returns_df).cumprod()

running_max = cumulative_returns.cummax()

drawdown = (
    cumulative_returns - running_max
) / running_max

max_drawdown = drawdown.min()

max_drawdown_df = pd.DataFrame({
    "Max_Drawdown": max_drawdown
})

max_drawdown_df.head()

,Max_Drawdown
amfi_code,
100016,-0.247344
100025,-0.043083
100033,-0.162172
101206,-0.112916
101207,-0.354469


In [11]:
max_drawdown_df.to_csv(
    "../reports/max_drawdown_report.csv"
)

In [12]:
benchmark = returns_df.mean(axis=1)

beta = {}
alpha = {}

for fund in returns_df.columns:

    covariance = np.cov(
        returns_df[fund],
        benchmark
    )[0,1]

    market_variance = np.var(benchmark)

    beta[fund] = covariance / market_variance

    alpha[fund] = (
        returns_df[fund].mean()*252
        -
        (
            0.06 +
            beta[fund] *
            (
                benchmark.mean()*252 - 0.06
            )
        )
    )

In [13]:
alpha_beta_df = pd.DataFrame({
    "Alpha": alpha,
    "Beta": beta
})

alpha_beta_df.head()

,Alpha,Beta
100016,-0.102460,0.789128
100025,-0.023164,0.060766
100033,0.113833,0.992453
101206,0.081600,0.737657
101207,-0.238298,2.880691


In [14]:
alpha_beta_df.to_csv(
    "../reports/alpha_beta_report.csv"
)

In [15]:
tracking_error = {}

for fund in returns_df.columns:

    tracking_error[fund] = (
        returns_df[fund] - benchmark
    ).std() * np.sqrt(252)

In [16]:
tracking_error_df = pd.DataFrame({
    "Tracking_Error": tracking_error
})

tracking_error_df.head()

,Tracking_Error
100016,0.144121
100025,0.046098
100033,0.187586
101206,0.144567
101207,0.251600


In [17]:
tracking_error_df.to_csv(
    "../reports/tracking_error_report.csv"
)